# Creating Python Code Files for SAS Intelligent Decisioning

This notebook demonstrates how to use the `CodeFile` class to upload Python code
files that are properly formatted for use with SAS Intelligent Decisioning.

## Overview

SAS Intelligent Decisioning requires Python code files to follow a specific format.

Here is a high-level summary of the formatting requirements:

- An `execute` function is required
- An `Output:` docstring listing output variables as the first line in the execute function
- A `DependentPackages:` docstring listing required packages at the top of the file including packages that are needed but are not built-in
- Must return standard Python data types


The `CodeFile` class validates and uploads properly formatted Python code to SAS Viya.

For more information about formating requirements for Python code files, see the [Rules
For Developing Python Code
Files](https://documentation.sas.com/?cdcId=edmcdc&cdcVersion=default&docsetId=edmug&docsetTarget=n04vfc1flrz8jsn1o5jblnbgx6i3.htm#n0jrohir6wzvd0n11omfautducm3)
in _SAS Intelligent Decisioning: User's Guide_.

## Prerequisites

- SAS Viya environment with SAS Intelligent Decisioning
- Appropriate permissions to create files in the target folder
- sasctl package installed
- Python code already formatted according to SAS Intelligent Decisioning requirements

## Setup: Connect to SAS Viya

In [ ]:
from sasctl import Session
from sasctl.pzmm import CodeFile
from sasctl.services import folders as folder_service


# Replace with your SAS Viya connection information
HOST = 'your-viya-host.com'
USERNAME = 'your-username'
PASSWORD = 'your-password'

# Create a session
sess = Session(HOST, USERNAME, PASSWORD, verify_ssl=False)
print(f"Connected to {HOST}")

try:
    folder_service.create_folder('ID_python_files', "/Public")
except Exception as error:
    print(f"Folder already exists. {error}")

## Example 1: Simple Code File

Here is a simple example that performs a basic calculation.

In [ ]:
# Define properly formatted Python code
simple_code = """
def execute(input_value):
    '''Output: score, category'''
    # Calculate a simple score
    score = input_value * 2 + 10
    category = 'High' if score > 50 else 'Low'
    return score, category
"""

# Upload the code file to SAS Viya
file_obj = CodeFile.write_id_code_file(
    code=simple_code,
    file_name='simple_calculator.py',
    folder='/Public/ID_python_files'
)

print(f"The file was uploaded successfully.")
print(f"File ID: {file_obj.id}")
print(f"File Name: {file_obj.name}")

## Example 2: Code File with API Call

Here is an example of how to create a code file that makes an API call to retrieve data.

In [ ]:
api_code = """
'''DependentPackages: requests'''
def execute(customer_id):
    '''Output: risk_score, status'''
    import requests
    import json

    # Make an API call
    url = f"https://api.example.com/data?id={customer_id}"
    response = requests.get(url)

    if response.status_code == 200:
        data = response.json()
        risk_score = data.get('risk_score', 0)
        status = 'Success'
    else:
        risk_score = -1
        status = 'Failed'
    
    return risk_score, status
"""

# Upload the code file
file_obj = CodeFile.write_id_code_file(
    code=api_code,
    file_name='risk_score_api.py',
    folder='/Public/ID_python_files'
)

print(f"The file was uploaded successfully.")
print(f"File ID: {file_obj.id}")
print(f"File name: {file_obj.name}")

## Example 3: Code with Multiple Dependencies

Here is an example of specifying multiple packages in the `DependentPackages` docstring.

In [ ]:
data_processing_code = """
'''DependentPackages: pandas, numpy'''
def execute(value1, value2, value3, threshold):
    '''Output: mean_value, std_value, result'''
    import pandas as pd
    import numpy as np

    # Create a simple dataframe
    data = pd.DataFrame({
        'values': [value1, value2, value3]
    })

    # Calculate statistics
    mean_value = float(np.mean(data['values']))
    std_value = float(np.std(data['values']))
    result = 'Pass' if mean_value > threshold else 'Fail'

    return mean_value, std_value, result
"""

# Upload the code file
file_obj = CodeFile.write_id_code_file(
    code=data_processing_code,
    file_name='data_processor.py',
    folder='/Public/ID_python_files'
)

print(f"This file was uploaded successfully: {file_obj.name}")

## Example 4: Reading Code from a File

Here is an example of reading Python code from an existing file.

In [ ]:
from pathlib import Path

# Create a properly formatted Python file
temp_code_file = Path('temp_code.py')
temp_code_file.write_text("""
def execute(income, assets, debt):
    '''Output: credit_score, decision, confidence'''
    # Business logic for credit decision
    credit_score = income * 0.3 + assets * 0.2 - debt * 0.5
    decision = 'Approved' if credit_score > 650 else 'Denied'
    confidence = min(credit_score / 850, 1.0)
    
    return credit_score, decision, confidence
""")

# Upload code from file (pass Path object)
file_obj = CodeFile.write_id_code_file(
    code=temp_code_file,
    file_name='credit_decision.py',
    folder='/Public/ID_python_files'
)

# Clean up
temp_code_file.unlink()

print(f"Code uploaded from file: {file_obj.name}")

## Example 5: Code File with No Parameters

Here is an example of creating code files that do not require input parameters.

In [ ]:
from sasctl.services import files as file_service
from sasctl.services import folders as folder_service

config_code = """
def execute():
    '''Output: current_date, environment, version'''
    import datetime

    # Get current configuration
    current_date = datetime.datetime.now().strftime('%Y-%m-%d')
    environment = 'production'
    version = '1.0.0'

    return current_date, environment, version
"""

# Check if file already exists and delete it.
# Warning: Deleting files might result in loss of important data or configurations.
# Ensure you have backups or that the file can be safely removed before proceeding.

file_name = 'config_info.py'
folder_path = '/Public/ID_python_files'

try:
        folder_obj = folder_service.get_folder(folder_path)

        file_filter = f"and(eq(name, '{file_name}'), eq(contentType, 'file'))"
        existing_file = folder_service.get(
            f"/folders/{folder_obj.id}/members",
            params={"filter": file_filter}
        )
        if len(existing_file) > 0:
            print(f"Warning: You are about to delete this file: {file_name}")
            print("This action might result in loss of sensitive data or configurations.")

            file_service.delete_file({"id": existing_file['uri'].split('/')[-1]})
            print(f"Deleted file: {file_name}")
except Exception as e:
    print(f"This file was not found: {file_name} {e}")


file_obj = CodeFile.write_id_code_file(
    code=config_code,
    file_name=file_name,
    folder=folder_path
)

print(f"Configuration code file created: {file_name}")

## Example 6: Disable Validation

Here is an example of skipping pre-upload validation.

**Note:** The file will still be uploaded even if it contains formatting errors.
The errors appear later when you try to use the file in a decision. You can
view the code file in SAS Intelligent Decisioning and validate it to check for errors.

In [ ]:
fast_code = """
def execute(input_a, input_b):
    '''Output: result'''
    result = input_a + input_b
    return result
"""

# Skip pre-upload validation for faster upload
# File is still created when there are formatting errors
file_obj = CodeFile.write_id_code_file(
    code=fast_code,
    file_name='fast_calculator.py',
    folder='/Public/ID_python_files',
    validate_code=False  # Skip pre-upload validation
)

print(f"File uploaded without pre-validation: {file_obj.name}")
print("Warning: If there are formatting errors, they will appear when you use the file in a decision.")

## Clean Up

Close the SAS Viya session when finished.

In [ ]:
# Close the session
sess.close()
print("Session closed")

## Additional Resources

- [SAS Intelligent Decisioning Documentation](https://documentation.sas.com/?cdcId=edmcdc&cdcVersion=default&docsetId=edmug&docsetTarget=n04vfc1flrz8jsn1o5jblnbgx6i3.htm)
- [Rules For Developing Python Code Files](https://documentation.sas.com/?cdcId=edmcdc&cdcVersion=default&docsetId=edmug&docsetTarget=n04vfc1flrz8jsn1o5jblnbgx6i3.htm#n0jrohir6wzvd0n11omfautducm3)
- [python-sasctl Documentation](https://sassoftware.github.io/python-sasctl/)